# 阶段 8：自动调度闭环（MLP → 防抖 → 帧边界切换）

目标：不需要人写寄存器，系统自己根据 MLP 判断的场景切换增强模式。

两个单元：

1. **短时观察**：在 notebook 里跑 30 秒，只打印，不切换 —— 先确认 MLP 判断稳定。
2. **正式运行**：用 `auto_schedule.py`（后台跑），notebook 不参与循环。

为什么正式演示要用 .py 而不是 notebook 的 `while True`：
Jupyter 的长循环会和前端通信抢线程，跑几分钟后经常出现“输出不刷新、内核像卡死”，
答辩现场非常危险。`auto_schedule.py` 是独立进程，notebook 只负责看日志。

In [ ]:
import os
import sys
import time

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib.ai_video_regs import AiVideoRegs

# 自动调度必须和采集用**同一档锁定曝光**：
# 训练数据是 736 行 + 增益 0x20 采的；开着自动曝光时 AEC/AGC 会改曝光/增益，
# 噪声一变 f_grad 就抬高十几个点，三类中心对不上，MLP 的判决会乱跳
# （2026-10-13 实测：自动曝光下 8 次切换全是 0<->2，逆光档一次都没出现）。
# 画面太暗/全黑时先改下面这个开关：False = 退回自动曝光（画面会亮起来，
# 但特征会偏移、三类中心对不上，MLP 的判决就不准了）。
USE_MANUAL_EXPOSURE = True
DEMO_EXPOSURE, DEMO_GAIN = 736, 0x20      # 采集时锁定的工作点，重采过就改这里
ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(
    ROOT, manual_exposure=(dict(exposure=DEMO_EXPOSURE, gain=DEMO_GAIN)
                           if USE_MANUAL_EXPOSURE else None))
vs.stop_all_channels(vdma0, vdma1)
regs = AiVideoRegs(ol.axis_video_proc_0)
assert regs.version_stage() >= 7, "本阶段需要阶段 7 的硬件"
vp.start_capture(vdma0, mode)
print("VERSION=0x%08X  FRAME_CNT=%d" % (regs.version(), regs.frame_cnt()))

## 0) 体检 + 场景响应测试（MLP 一直是同一个值时先跑这里）

MLP 的三个输入是硬件算出来的三个整数，不是原始画面（权威定义见 `docs/01_register_map.md`）：

| 类别 | 场景 | f_luma | f_luma2 | f_grad |
|------|------|--------|---------|--------|
| 0 | 暗光（镜头被挡死 / 全黑） | ≈ 10 | ≈ 1 | ≈ 25 |
| 1 | 逆光（对着窗户，明暗对比大） | ≈ 67 | ≈ 39 | ≈ 30 |
| 2 | 常规 / 细节（正常室内 + 移动物体） | ≈ 109 | ≈ 55 | ≈ 35 |

表里的数是 `data/features.csv` 里每类 200 帧的实测均值。

所以「MLP 一直是 0」是两种完全不同的情况，必须先分开：

* **画面真的黑**：特征就停在 (10, 1, 25) 附近 —— 判 0 是**对的**，不是 bug；
* **链路卡住**：特征不随场景变，或者 `MLP_START_CNT` / `MLP_DONE_CNT` 不涨。

下面两格：先跑第一格记基线，然后**改变场景**（摘镜头盖 / 开灯 / 对着窗户 / 把亮的东西放到镜头前），再跑第二格 —— 它会直接告诉你卡在哪一环。

In [ ]:
# 体检：把 MLP 判决、三个特征、四个计数器一次读全。
# 三类中心 / 名称 —— 从 data/features.csv 现算（和报告、lib/adaptive.py、auto_schedule.py 同一来源）。
# 别写死：重采数据之后中心会动（旧中心 9.8/67.4/109.3 → 现中心 9.7/66.4/96.5）。
# 写死了就会拿旧目标去解释距离，看着"差一点点"其实已经偏了 —— 而 scripts 那边
# 的门限是按 features.csv 现算的，两边口径不一致最容易把人绕晕。
from lib import adaptive as ad
CLASS_MEAN = {k: tuple(ad.CLASS_MEAN[k]) for k in (0, 1, 2)}
CLASS_NAME = {k: ad.CLASS_NAME[k] for k in (0, 1, 2)}
print("三类中心来源：%s" % ad.CLASS_STATS_SOURCE)
for _k in (0, 1, 2):
    print("  %d %-6s f=(%5.1f,%5.1f,%5.1f)" % ((_k, CLASS_NAME[_k]) + CLASS_MEAN[_k]))

def health(tag=""):
    st = regs.raw_stats()
    f1, f2, f3 = regs.mlp_features_from_raw(st)
    luma, contrast, sharp = regs.features_from_raw(st)
    dist = {k: ((f1-a)**2 + (f2-b)**2 + (f3-c)**2) ** 0.5
            for k, (a, b, c) in CLASS_MEAN.items()}
    near = min(dist, key=dist.get)
    h = dict(f=(f1, f2, f3), frame=regs.frame_cnt(), vblank=regs.vblank_cnt(),
             start=regs.mlp_start_cnt(), done=regs.mlp_done_cnt(),
             mlp=regs.mlp_output(), mode=regs.mode_sel())
    print("---- 体检 %s ----" % tag)
    print("VERSION=0x%08X (阶段%d)  MODE_SEL=%d  MLP_OUTPUT=%d  最近类别=%d(%s)"
          % (regs.version(), regs.version_stage(), h["mode"], h["mlp"],
             near, CLASS_NAME[near]))
    print("FRAME_CNT=%d  VBLANK_CNT=%d  PIXEL_CNT=%d  MLP_START=%d  MLP_DONE=%d"
          % (h["frame"], h["vblank"], st["pixel_cnt"], h["start"], h["done"]))
    print("MLP 输入 f=(%d, %d, %d)   luma=%.1f contrast=%.1f sharpness=%.1f"
          % (f1, f2, f3, luma, contrast, sharp))
    print("到三类中心距离: " + "  ".join("%s=%.1f" % (CLASS_NAME[k], v)
          for k, v in sorted(dist.items())))
    return h

h1 = health("基线")

In [ ]:
# 换个场景再跑这一格：摘镜头盖 / 开灯 / 对着窗户 / 亮屏怼到镜头前。
h2 = health("换场景后")

near2 = min(CLASS_MEAN,
            key=lambda k: sum((x-y)**2 for x, y in zip(h2["f"], CLASS_MEAN[k])))
print()
print("---- 两次读数之差 ----")
print("FRAME_CNT +%d   VBLANK_CNT +%d   MLP_START +%d   MLP_DONE +%d"
      % (h2["frame"]-h1["frame"], h2["vblank"]-h1["vblank"],
         h2["start"]-h1["start"], h2["done"]-h1["done"]))
print("f %s -> %s" % (h1["f"], h2["f"]))
print()
if h2["frame"] == h1["frame"]:
    print(">>> 采集链停了：FRAME_CNT 没动。重跑第 2 格，再查摄像头 / VDMA。")
elif h2["f"] == h1["f"]:
    print(">>> 视频在流、但特征一个数都没变：画面本身没变（镜头被挡死，或者对着全黑）。")
    print("    摘掉镜头盖 / 开灯 / 拿亮的东西放到镜头前，再跑一次本格。")
elif h2["start"] == h1["start"]:
    print(">>> 特征在变、但 MLP_START_CNT 不涨：硬件启动握手的问题，得改 HDL 重出 bitstream。")
elif h2["done"] == h1["done"]:
    print(">>> MLP 启动了但没完成（MLP_DONE_CNT 不涨）：查 MLP 的 ap_clk / ap_rst。")
else:
    print(">>> 整条链都活着：MLP_OUTPUT 只是老实反映了当前画面。")
    print("    当前最近类别 = %d(%s)。要看到自动切换，就得让画面亮度真的变："
          % (near2, CLASS_NAME[near2]))
    print("    全黑 f_luma≈10(判0) / 逆光≈67(判1) / 常规细节≈109(判2)。")

## 1) 短时观察：只看不切

跑起来之后依次遮镜头 / 逆光 / 移动物体，观察 `mlp` 是否稳定地停留在 0/1/2。
如果它一直在跳，先把 `debounce` 调大（脚本里 `--debounce`）；
如果它一直**不动**，先回上面 `0) 体检` 那两格 —— 多半是画面本身没变（全黑 / 没开灯）。

In [ ]:
def observe(seconds=30.0, poll=0.2):
    """盯一段时间，把 MLP 判决和三个特征一起打出来。

    只打 MLP_OUTPUT 分不清「画面真的黑」和「链路卡住」，
    所以这里连 f=(f_luma, f_luma2, f_grad) 和 FRAME_CNT 一起看。
    """
    t0 = time.time()
    rows = []
    tick = 0.0
    while time.time() - t0 < seconds:
        t = time.time() - t0
        f = regs.mlp_features()
        mlp = regs.mlp_output()
        fc = regs.frame_cnt()
        rows.append((round(t, 2), mlp, f, fc))
        if t >= tick:                      # 每秒打一行，免得刷屏
            tick = t + 1.0
            print("t=%5.1fs mlp=%d f=(%3d,%3d,%3d) FRAME_CNT=%d"
                  % (t, mlp, f[0], f[1], f[2], fc))
        time.sleep(poll)

    counts = {}
    for _, m, _, _ in rows:
        counts[m] = counts.get(m, 0) + 1
    fl = [r[2][0] for r in rows]
    fg = [r[2][2] for r in rows]
    print("MLP 分布:", counts)
    print("f_luma %d~%d   f_grad %d~%d   FRAME_CNT %d -> %d"
          % (min(fl), max(fl), min(fg), max(fg), rows[0][3], rows[-1][3]))
    if max(fl) - min(fl) == 0 and max(fg) - min(fg) == 0:
        print("注意：这 %.0f 秒里特征一个数都没变 —— 画面没变，MLP 自然也不变。" % seconds)
    return rows

rows = observe(30.0)

## 2) 正式运行：用独立进程跑调度

先把上面那格停掉（点 ■），再运行下面这格。它会**后台**启动调度脚本，
并把日志写到 `/home/xilinx/sched.log`。

启动前必须先把 notebook 占用的 overlay 通道放掉，否则两个进程抢同一个 DMA。

三个坑（第一次跑都踩过）：

1. **用 `sys.executable` 启动，不要写死 `python3`**：板子上 `python3` 可能是系统解释器，
   里面没有 pynq 模块，脚本会秒退；`> /dev/null` 又把错误一起吞掉，看起来像“什么都没发生”。
2. **别把 stderr 丢掉**：脚本自己的报错要留在 `sched_stdout.log` 里，出错才查得动。
3. **等 10 秒再 tail**：加载 overlay + 配 253 个摄像头寄存器要好几秒，只等 2 秒看不到东西。

另外：真机上 MLP 会在相邻帧之间偶发抖动，光靠 `--debounce` 防不住来回跳
（实测 1.6 秒里切了 4 次）。脚本现在多了一道 `--min-hold` 时间闸，两次切换至少隔这么久。

还有两个开关是给“notebook 已经初始化过”这种情况用的：

* `--no-camera`：不再重配摄像头。传感器寄存器写一次就留在传感器里，不需要第二个进程再写；
  再写一遍偶尔会抛 `Could not send I2C data`（2026-10-03 就这样把调度脚本弄挂过）。
* `--keep-video`：不碰 VDMA 通道，避免把上面那格刚打开的 HDMI 显示通路拆掉。
* `--heartbeat 5`：每 5 秒往日志里写一行 `mlp / 模式 / FRAME_CNT / 特征`。
  事后一看日志就知道「MLP 不动」是画面没变还是链路卡住 —— 这是本次新加的诊断口子。

In [ ]:
# 打开 HDMI 显示（演示必须；不打开的话从阶段 3 起一直是黑屏）
#
# 采集和显示是两条链：
#   OV5640 -> v_demosaic -> axis_video_proc -> VDMA0 S2MM -> DDR
#   DDR -> VDMA0 MM2S -> VDMA1 S2MM -> DDR -> VDMA1 MM2S -> v_axi4s_vid_out -> rgb2dvi -> HDMI
# 前面各阶段只调了 vp.start_capture()（只启动采集），所以 HDMI 一直没有数据 = 黑屏，
# 这是正常的、不是坏了。演示时用下面这个函数把两条链都启动（它内部先 tie 再 start）。
vp.start_hdmi_passthrough(vdma0, vdma1, mode)
for name, ch in (("vdma0.write(MM2S)", vdma0.writechannel),
                 ("vdma0.read(S2MM)",  vdma0.readchannel),
                 ("vdma1.write(MM2S)", vdma1.writechannel),
                 ("vdma1.read(S2MM)",  vdma1.readchannel)):
    print("%-18s running=%s" % (name, ch.running))
print("HDMI 现在应该是实时画面，并且会跟着 MODE_SEL 变")

# 下一步启动调度脚本要用的基地址：脚本用 MMIO 直接戳这个 IP 的寄存器，
# 这样它不加载 overlay、不碰 VDMA、不碰 I2C —— 就不会和 notebook 抢任何东西。
# 注意：PYNQ 的 ip_dict 键名是 'phys_addr'（PYNQ 2.x / 3.x 都是），
#       写成 'phys_address' 会 KeyError —— 2026-10-03 就在这上面卡过一轮。
BASE = ol.ip_dict["axis_video_proc_0"]["phys_addr"]
print("axis_video_proc 基地址 = 0x%X（下一格要用）" % BASE)

In [ ]:
import subprocess
import sys

LOG    = "/home/xilinx/sched.log"          # 调度脚本自己的日志
STDOUT = "/home/xilinx/sched_stdout.log"   # 脚本的 stdout/stderr（报错看这里）
SCRIPT = os.path.join(ROOT, "stage08_auto_schedule", "auto_schedule.py")

# ---- 启动前先清理 ----------------------------------------------------------
# 为什么：两个实例同时写同一个日志会让心跳交错，而且后启动的那个接手时
# MODE_SEL 可能已经等于当前类别 → 收尾行变成“共切换 0 次”，把上一轮的
# 好结果盖掉。2026-10-05 真踩过：第 10 格点了两次。
left = subprocess.run(["pgrep", "-f", "auto_schedule.py"],
                      capture_output=True, text=True).stdout.split()
if left:
    print("发现残留实例 %s，先结束它们" % left)
    subprocess.run(["pkill", "-f", "auto_schedule.py"])
    for _ in range(20):
        alive = subprocess.run(["pgrep", "-f", "auto_schedule.py"],
                               capture_output=True, text=True).stdout.strip()
        if not alive:
            break
        time.sleep(0.5)

# 清空旧日志：这样 sched.log 里只有本轮，最后一行就是验收要看的收尾行
open(LOG, "w").close()
open(STDOUT, "w").close()

# 关键：用 sys.executable（= 当前 notebook 内核的 python，一定带着 pynq），
# 不要写死 python3 —— 板子上的 python3 可能是系统解释器，没有 pynq 模块，
# 脚本会立刻退出；如果同时把输出重定向到 /dev/null，就完全看不出发生过什么。
#
# --base：脚本用 MMIO 直接读写这个 IP 的寄存器（不加载 overlay / 不碰 VDMA / 不碰 I2C），
#         所以 notebook 这边的采集+显示通路不会被它动到，也不会再出现
#         "Could not send I2C data" 那种两个进程抢总线的失败。
cmd = [sys.executable, SCRIPT,
       "--base", "0x%X" % BASE,
# 阶段 12：加上 --adaptive 就会在切换模式的同时按实时特征连续写 PARAM。
# 不加也能跑（固定 PARAM=192，等于阶段 7 的行为）；硬件阶段号 < 12 时这项会被自动忽略。
# 想强制关掉自适应，把下面这一行注释掉即可。
       "--adaptive",
       "--duration", "60", "--debounce", "8", "--min-hold", "2.0",
       "--heartbeat", "5", "--log", LOG]
print("启动:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=open(STDOUT, "a"), stderr=subprocess.STDOUT)
print("pid =", proc.pid)
print("（本格会等到收尾行出现再打印，约 60~80 秒，期间不要重复点）")

# 等它真的跑完：出现收尾行就停，最多等 110 秒。
# 为什么不用 time.sleep(10)：PYNQ 上光 import pynq + numpy 就要十几秒，
# 固定等 10 秒会看到空日志，容易让人以为没跑起来、再点一次 —— 那样就是两个实例了。
t0 = time.time()
while time.time() - t0 < 110:
    txt = open(LOG).read()
    if "共切换" in txt:
        break
    time.sleep(2)

print("---- sched.log ----")
print(open(LOG).read())
print("---- 脚本自己的 stdout/stderr（末尾 1500 字符）----")
print(open(STDOUT).read()[-1500:])

In [ ]:
# 隔一会儿看日志（重复运行本单元即可，不用等 --duration 到点）
LOG    = globals().get("LOG", "/home/xilinx/sched.log")
STDOUT = globals().get("STDOUT", "/home/xilinx/sched_stdout.log")

print(subprocess.run(["tail", "-n", "20", LOG], capture_output=True, text=True).stdout)
print("---- 脚本自己的输出（有报错就看这里）----")
print(subprocess.run(["tail", "-n", "20", STDOUT], capture_output=True, text=True).stdout)

# 想立刻结束、不等 60 秒：
# subprocess.run(["pkill", "-f", "auto_schedule.py"])
# print(subprocess.run(["tail", "-n", "40", LOG], capture_output=True, text=True).stdout)

## 结论

* 日志**最后一行**是 `共切换 N 次，因 min_hold(2.0s) 被挡下 M 次，运行 60.x 秒` 且 **N > 0** → 阶段 8 完成。
* 日志里出现 `切换模式 x -> y` 且画面确实变了 → 说明 MLP → 防抖 → 帧边界这条闭环是通的。
* 切换没有撕裂 → 说明 `wait_frame_boundary()` + 硬件端帧边界重采样起作用了。
* 日志里的 `心跳 ...` 行带着 f=(f_luma, f_luma2, f_grad)：特征一直不变 = 画面没变，
  MLP 一直判同一个类是对的，不是代码问题。
* 正式演示前记得 `pkill -f auto_schedule.py`，避免和阶段 11 的演示脚本抢资源。